## Annexure 1: Speaker Count Data 

#### Re: AfricaNLP Gap Analysis Memo 

This Annexure contains the code for cleaning the first three of the datasets we employed in our analysis:

I. The merged dataset of the latest survey round (2023) of the Afrobarometer as available [here](https://www.afrobarometer.org/data/merged-data/). Its codebook is [here](https://www.afrobarometer.org/wp-content/uploads/2025/07/AB_R9.MergeCodebook_25Jun24.final_.pdf). This covers 39 African countries (Angola, Benin, Botswana, Burkina Faso, Cabo Verde, Cameroon, CongoBrazzaville, Côte d'Ivoire, Eswatini, Ethiopia, Gabon, Gambia, Ghana, Guinea, Kenya, Lesotho, Liberia, Madagascar, Malawi, Mali, Mauritania, Mauritius, Morocco, Mozambique, Namibia, Niger, Nigeria, São Tomé and Príncipe, Senegal, Seychelles, Sierra Leone, South Africa, Sudan, Tanzania, Togo, Tunisia, Uganda, Zambia, and Zimbabwe) thereby covering a substantial portion of the African continent.

II. A list of the number of speakers (worldwide) of 7,130 Languages , made available in 2026 on Huggingface by Luke Steuber [here](https://huggingface.co/datasets/lukeslp/world-languages), where speaker counts have been compiled from the Joshua Project[(https://joshuaproject.net/data#:~:text=Joshua%20Project%20builds%20a%20comprehensive,church%20engagement%20across%20the%20globe)] a Christian organization and aggregator of global ethno-linguistic data (hereinafter, "JP dataset"). 


III. To assign unique identifiers to African languages, the Huggingface database uses ISO-639-1 ('two letter') and ISO-639-3 ('three-letter') language codes (listed in this [repo](https://github.com/huggingface/huggingface.js/blob/main/packages/languages/src/languages_iso_639_3.ts)). These ISO codes are the primary international standards for language codes followed across the world, including in the second dataset above and for tagging NLP datasets on Huggingface. However, this is not easily mapped to the Afrobarometer dataset, because the name of a language as per its iso3 code, as listed in the Huggingface lists, does not map in most cases to a survey respondent's self-identified language. Further, there are often inconsistences across datasets in how African languages are mapped to ISO codes in general. 

> Accordingly, a number of procedures are applied below to attempt to map the iso3 code to a language, including: 

(1) at a first pass, over a 100 ISO codes were manually mapped using a rapid internet search

(3) at a second level, a large language model (GPT 5.2) was used to map HuggingFace ISO-639-3 codes to the names of languages in the Afrobarometer survey data, which relied on its training data to exercise judgement. Only instances where it was highly confident were retained; a manual cross-check of a random sample was used to update the manual mapping above. 

At the end of this exercise, I am left with 195 of the 657 languages unmatched to ISO-639-1 or ISO-639-3 codes, with a weighted share of <10%, at which point I assume the resulting ISO-3 mappings are sufficiently reliable to proceed with. 

At the end of this notebook, we receive a dataset ("matched") that contains a language, its country, its iso-639-1 or iso-639-3 code, its weighted in-country share per the survey sample, and its estimated speaker count (global, not in-country). We also receive a similar dataset ("unmatched") with the same columns other than the (missing) codes and population statistics (as there was no iso-3 code to merge population statistics with). 


### Table of Contents:

1. [Setup](#1-setup--dataset-loading)

2. [Matching HuggingFace ISO-639-3 codes to ABM dataset](#2-matching-iso-639-3-codes-to-abm-dataset)

3. [Merging Datasets](#3-merging-datasets)

## 1. Setup & Dataset Loading

In [454]:
# Uncomment to install dependencies
# !pip install pandas numpy rapidfuzz dotenv

In [455]:
import pandas as pd
import numpy as np
import os, json, re, requests
from pathlib import Path
from rapidfuzz import process, fuzz
from dotenv import load_dotenv
from io import StringIO

pd.set_option("mode.copy_on_write", True)

In [456]:
afrobm_countries = [
    "Angola",
    "Benin",
    "Botswana",
    "Burkina Faso",
    "Cabo Verde",
    "Cameroon",
    "CongoBrazzaville",
    "Côte d'Ivoire",
    "Eswatini",
    "Ethiopia",
    "Gabon",
    "Gambia",
    "Ghana",
    "Guinea",
    "Kenya",
    "Lesotho",
    "Liberia",
    "Madagascar",
    "Malawi",
    "Mali",
    "Mauritania",
    "Mauritius",
    "Morocco",
    "Mozambique",
    "Namibia",
    "Niger",
    "Nigeria",
    "São Tomé and Príncipe",
    "Senegal",
    "Seychelles",
    "Sierra Leone",
    "South Africa",
    "Sudan",
    "Tanzania",
    "Togo",
    "Tunisia",
    "Uganda",
    "Zambia",
    "Zimbabwe",
]

In [457]:
# Base directories
BASE_DIR = Path.cwd()

# Subdirectories
RAW = BASE_DIR / "raw_data"
IN_USE = BASE_DIR / "in_use_data"
CLEAN = BASE_DIR / "clean_data"

# Datasets
AFRO = (
    RAW / "afro.sav"
)  # list of languages identified as "spoken at home" by Afrobarometer survey respondents
HF_ISO639_1 = (
    RAW / "hf_iso_639_1.ts"
)  # list of languages identified with ISO_639_1 codes on Huggingface
HF_ISO639_3 = (
    RAW / "hf_iso_639_3.ts"
)  # list of languages identified with ISO_639_3 codes on Huggingface
JOSHUA = (
    RAW / "lang-pop.json"
)  # global language speaker count statistics from the Joshua project

In [458]:
# loading afrobarometer survey data
afro_raw = pd.read_spss(AFRO)

# filtering down to necessary columns
afro_selection = afro_raw[
    [
        "RESPNO",
        "COUNTRY",
        "REGION",
        "LOCATION.LEVEL.1",
        "Q2",
        "Q2OTHER",
        "Combinwt_new_hh",
        "NOCALL_1",
    ]
].rename(columns={"Combinwt_new_hh": "weight"})

# filtering out responses where no interview took place
afro = afro_selection[afro_selection["NOCALL_1"] == "Interview was successful"].copy()

# creating unified language column
afro["language"] = np.where(afro["Q2"] == "Other", afro["Q2OTHER"], afro["Q2"])

afro = afro.rename(
    columns={"COUNTRY": "country", "REGION": "region", "LOCATION.LEVEL.1": "location"}
)
afro = afro[["country", "region", "location", "language", "weight"]]

# check for blank language values
print(afro["language"].isna().sum())

# as all language values exist, identifying rows where respondents did not disclose what languages they spoke at home
afro["no_language"] = afro["language"].isin(["refused", "don't know", "Don't know", ""])
print(afro[afro["no_language"] == True])

# Weighted share
no_response_share = afro.loc[afro["no_language"], "weight"].sum() / afro["weight"].sum()
print(no_response_share)

0
            country         region    location    language    weight  \
4242   Burkina Faso  Hauts-Bassins  KENEDOUGOU  Don't know  0.530233   
25410        Malawi       Southern       Zomba  Don't know  0.756788   
31395    Mozambique       Zambézia   NAMACURRA  Don't know  1.405940   
32065    Mozambique       Zambézia     MILANGE  Don't know  1.167554   
46368          Togo           KARA       KERAN  Don't know  1.283055   

       no_language  
4242          True  
25410         True  
31395         True  
32065         True  
46368         True  
0.00013129656304033008


> As the weighted share of "no language reported" is ~0.001%, we can proceed with dropping these values from the rest of the dataset 

In [459]:
# dropping values
afro = afro.loc[
    ~afro["language"].isin(["don't know", "Don't know", "refused", "Refused", ""])
][["country", "region", "location", "language", "weight" ""]]

In [460]:
# created list of languages by country
country_languages = (
    afro[["country", "language"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["country", "language"])
)
country_languages["language_clean"] = (
    country_languages["language"].str.strip().str.lower()
)
country_languages

,country,language,language_clean
599,Angola,Chokwe,chokwe
264,Angola,Fiote/Ibinda,fiote/ibinda
63,Angola,Kikongo,kikongo
40,Angola,Kimbundu,kimbundu
493,Angola,LINGALA,lingala
...,...,...,...
53435,Zimbabwe,Portuguese,portuguese
52244,Zimbabwe,Shona,shona
52515,Zimbabwe,Tonga,tonga
52588,Zimbabwe,Venda,venda


> A total of 659 unique languages are reported by Afrobarometer survey respondents

In [461]:
# loading huggingface language lists
HF_ISO639_1 = "https://raw.githubusercontent.com/huggingface/huggingface.js/refs/heads/main/packages/languages/src/languages_iso_639_1.ts"
text_1 = requests.get(HF_ISO639_1).text
matches_1 = re.findall(
    r'code:\s*"([^"]+)",\s*name:\s*"([^"]+)",\s*nativeName:\s*"([^"]+)"', text_1
)
hf_iso_1 = pd.DataFrame(matches_1, columns=["iso1", "language", "language_alt"])


HF_ISO639_3 = "https://raw.githubusercontent.com/huggingface/huggingface.js/refs/heads/main/packages/languages/src/languages_iso_639_3.ts"
text_3 = requests.get(HF_ISO639_3).text
matches_3 = re.findall(r'code:\s*"([^"]+)",\s*name:\s*"([^"]+)"', text_3)
hf_iso_3 = pd.DataFrame(matches_3, columns=["iso3", "language"])

# removing ISO
hf_iso_3 = hf_iso_3[~hf_iso_3["iso3"].isin(["eee", "uuu"])]
print(len(hf_iso_1), len(hf_iso_3))

184 7914


> There are a total of 189 ISO-639-1 language codes and 7914 ISO-639-3 language codes used by Huggingface, after the removal of two ISO-639-3 codes that are not mapped to any language.

In [462]:
# loading joshua project dataset
with open(JOSHUA) as f:
    df_lang = json.load(f)

df_lang = pd.DataFrame(df_lang)

pop = (
    df_lang[["iso_639_3", "name", "speaker_count"]]
    .rename(columns={"iso_639_3": "iso3"})
    .copy()
)

# dropping rows with Null values under speaker_count
pop_counts = pop.dropna(subset=["iso3", "speaker_count"])
pop_counts["speaker_count_clean"] = pop_counts["speaker_count"].str.get("count")

# reshaping for country-level observations
pop_counts["countries"] = (
    pop_counts["speaker_count"].str.get("metadata").str.get("countries")
)
pop_counts = pop_counts.explode("countries")

# remapping names with Afrobarometer names
pop_counts["countries"] = pop_counts["countries"].replace(
    {
        "Congo, Republic of the": "CongoBrazzaville",
        "Congo, Democratic Republic of": "CongoKinshasa",  # if needed
    }
)

# renaming columns
pop_counts = pop_counts[["iso3", "name", "speaker_count_clean", "countries"]].rename(
    columns={
        "countries": "country",
        "name": "language",
        "speaker_count_clean": "speakers",
    }
)
pop_counts = pop_counts[["country", "language", "iso3", "speakers"]]
pop_counts

,country,language,iso3,speakers
1,Ethiopia,Aari,aiw,558000
3,Papua New Guinea,Abadi,kbt,4200
4,Malaysia,Abai Sungai,abf,1600
5,Nigeria,Obanliku,bzy,143000
6,Nigeria,Abanyom,abm,37000
...,...,...,...,...
7128,Malawi,Zulu,zul,15400100
7128,Lesotho,Zulu,zul,15400100
7128,South Africa,Zulu,zul,15400100
7128,Zimbabwe,Zulu,zul,15400100


## 2. Matching HuggingFace ISO-639-3 codes to ABM dataset

In [463]:
# stripping to enable mapping
hf_iso_1["lang_clean"] = hf_iso_1["language"].str.strip().str.lower()
hf_iso_3["lang_clean"] = hf_iso_3["language"].str.strip().str.lower()

In [464]:
mapping_hf_iso1 = country_languages.merge(
    hf_iso_1,
    left_on="language_clean",
    right_on="lang_clean",
    how="left",
    indicator=True,
)
mapping_hf_iso1

,country,language_x,language_clean,iso1,language_y,language_alt,lang_clean,_merge
0,Angola,Chokwe,chokwe,NaN,NaN,NaN,NaN,left_only
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN,NaN,NaN,left_only
2,Angola,Kikongo,kikongo,NaN,NaN,NaN,NaN,left_only
3,Angola,Kimbundu,kimbundu,NaN,NaN,NaN,NaN,left_only
4,Angola,LINGALA,lingala,ln,Lingala,Lingála,lingala,both
...,...,...,...,...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,pt,Portuguese,Português,portuguese,both
728,Zimbabwe,Shona,shona,sn,Shona,chiShona,shona,both
729,Zimbabwe,Tonga,tonga,to,Tonga,faka Tonga,tonga,both
730,Zimbabwe,Venda,venda,ve,Venda,Tshivenḓa,venda,both


In [465]:
mapping_hf_iso3 = country_languages.merge(
    hf_iso_3,
    left_on="language_clean",
    right_on="lang_clean",
    how="left",
    indicator=True,
)
mapping_hf_iso3

,country,language_x,language_clean,iso3,language_y,lang_clean,_merge
0,Angola,Chokwe,chokwe,cjk,Chokwe,chokwe,both
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN,NaN,left_only
2,Angola,Kikongo,kikongo,NaN,NaN,NaN,left_only
3,Angola,Kimbundu,kimbundu,kmb,Kimbundu,kimbundu,both
4,Angola,LINGALA,lingala,lin,Lingala,lingala,both
...,...,...,...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por,Portuguese,portuguese,both
728,Zimbabwe,Shona,shona,sna,Shona,shona,both
729,Zimbabwe,Tonga,tonga,NaN,NaN,NaN,left_only
730,Zimbabwe,Venda,venda,ven,Venda,venda,both


> After the mapping, we have 649 ISO-1 codes and 516 ISO-3 codes that are unmatched. This is expected, as several African languages do not have ISO-639-1 codes, and, further, their names as reported by survey respondents are known by different names on the HuggingFace ISO-639-3 lists. To address this, manual maps are created based on individual research of those unmatched.  

> However, it was difficult to be confident, due to many regional variants in how languages are spelt or identifed. For instance, several language names from respondents in Nigeria appear to be town names; Nigeria alone has over 520 languages as per online sources. Further, there appear to be cases where the English version of a language name, when converted from its original script, is not consistently recorded (e.g., MENTIGNA). Consequently, Such values are hard to reconcile without directly asking native speakers to confirm.

In [466]:
unmatched_iso1 = mapping_hf_iso1[mapping_hf_iso1["_merge"] == "left_only"]
unmatched_iso1

,country,language_x,language_clean,iso1,language_y,language_alt,lang_clean,_merge
0,Angola,Chokwe,chokwe,NaN,NaN,NaN,NaN,left_only
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN,NaN,NaN,left_only
2,Angola,Kikongo,kikongo,NaN,NaN,NaN,NaN,left_only
3,Angola,Kimbundu,kimbundu,NaN,NaN,NaN,NaN,left_only
5,Angola,Muhumbi,muhumbi,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...
723,Zimbabwe,Korekore,korekore,NaN,NaN,NaN,NaN,left_only
724,Zimbabwe,Manyika,manyika,NaN,NaN,NaN,NaN,left_only
725,Zimbabwe,Ndau,ndau,NaN,NaN,NaN,NaN,left_only
726,Zimbabwe,Ndebele,ndebele,NaN,NaN,NaN,NaN,left_only


In [467]:
manual_iso1_map = {
    "arabe / hassaniya": "ar",
    "hassania arabic": "ar",
    "moroccan arabic": "ar",
    "sudanese arabic": "ar",
    "tunisian arabic": "ar",
    "afan oromo": "om",
    "haoussa": "ha",
    "ibo": "ig",
    "ewe/anlo": "ee",
    "ewé": "ee",
    "luganda": "lg",
    "kinyarwanda": "rw",
    "runyarwanda": "rw",
    "runyankore": "ny",
    "nyanja": "ny",
    "chinyanja": "ny",
    "chewa": "ny",
    "chichewa": "ny",
    "sesotho": "st",
    "sotho": "st",
    "setswana": "tn",
    "siswati": "ss",
    "swazi": "ss",
    "ndebele": "nd",
    "otjiherero": "hz",
    "seherero": "hz",
    "wolof / lebou": "wo",
    "fulani": "ff",
    "fulfudé": "ff",
    "fulfuldé": "ff",
    "fufuldé": "ff",
    "fullah": "ff",
    "peulh": "ff",
    "peulh / fulfude": "ff",
    "poular": "ff",
    "pulaar/toucouleur": "ff",
    "pular": "ff",
    "kiswahili": "sw",
    "shona": "sn",
    "tigrigna": "ti",
    "tigirigna": "ti",
    "somaligna": "so",
}

mapping_hf_iso1["iso_1_manual"] = mapping_hf_iso1["language_clean"].map(manual_iso1_map)
mapping_hf_iso1["iso1"] = mapping_hf_iso1["iso1"].fillna(
    mapping_hf_iso1["iso_1_manual"]
)
mapping_hf_iso1 = mapping_hf_iso1[
    ["country", "language_x", "language_clean", "iso1"]
].rename(columns={"language_x": "language"})
mapping_hf_iso1

,country,language,language_clean,iso1
0,Angola,Chokwe,chokwe,NaN
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN
2,Angola,Kikongo,kikongo,NaN
3,Angola,Kimbundu,kimbundu,NaN
4,Angola,LINGALA,lingala,ln
...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,pt
728,Zimbabwe,Shona,shona,sn
729,Zimbabwe,Tonga,tonga,to
730,Zimbabwe,Venda,venda,ve


In [468]:
unmatched_iso3 = mapping_hf_iso3[mapping_hf_iso3["_merge"] == "left_only"]
unmatched_iso3

,country,language_x,language_clean,iso3,language_y,lang_clean,_merge
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN,NaN,left_only
2,Angola,Kikongo,kikongo,NaN,NaN,NaN,left_only
5,Angola,Muhumbi,muhumbi,NaN,NaN,NaN,left_only
6,Angola,Nganguela,nganguela,NaN,NaN,NaN,left_only
7,Angola,Nhaneca,nhaneca,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...
720,Zambia,Ushi,ushi,NaN,NaN,NaN,left_only
723,Zimbabwe,Korekore,korekore,NaN,NaN,NaN,left_only
726,Zimbabwe,Ndebele,ndebele,NaN,NaN,NaN,left_only
729,Zimbabwe,Tonga,tonga,NaN,NaN,NaN,left_only


In [469]:
manual_iso3_map = {
    "Muhumbi": "xum",
    "KOURA": "kro",
    "Sengologa": "xkg",
    "Fulfudé": "fuf",
    "Fulse": "mos",
    "Gourounsi": "gux",
    "SASSALA": "sld",
    "Sénoufo": "seb",
    "TOUSSIAN": "wib",
    "ZAORÉ": "mos",
    "Bafang": "nnz",
    "Bamileke": "bai",
    "Bula": "bub",
    "Gbaya": "gya",
    "Teke": "teg",
    "Kroumen": "ted",
    "WORODOUGOU": "jud",
    "AGEW": "ahg",
    "Adergigna orHarari": "har",
    "BAINUNKA": "bcz",
    "TURKA": "tuz",
    "ACHODE": "ach",
    "BANDA": "bpd",
    "BEM": "bem",
    "BULISA": "bwu",
    "BUSANGA": "bvx",
    "GRUMA": "gma",
    "GUAN": "gua",
    "Gruni": "gur",
    "Grusi": "gsl",
    "MOAR": "moa",
    "MOLI": "gas",
    "Nabt": "nab",
    "Baga": "bga",
    "Kono": "kno",
    "Manian": "myq",
    "MALAKOTE": "mlk",
    "SHENG'": "shr",
    "Krahn": "krw",
    "Kru": "kru",
    "Dogon": "dts",
    "Samogo": "smo",
    "Senufo": "seb",
    "Sonrhaï": "son",
    "Amazigh": "zgh",
    "CHIBARUE": "brw",
    "MUNIGA": "mhm",
    "TAKWANE": "tke",
    "KHWEDAM": "khi",
    "MBALANGWE": "mck",
    "NYEMBA": "nba",
    "OLUDHIMBA": "dhm",
    "OSHINGANGELA": "gng",
    "SAN LANGUAGE": "",
    "Thimbukushu": "mhw",
    "AGBOR": "agb",
    "ANIOCHA": "nne",
    "ANKWOI": "ank",
    "BOKOBARU": "bus",
    "BURUM": "bux",
    "JIBU": "jib",
    "KAMI": "kmi",
    "KONA": "klk",
    "MUYE": "mye",
    "OKON": "oko",
    "CREOLE": "crp",
    "Mandinka/Bambara": "man",
    "Nubian language": "nub",
    "Creolo": "cri",
    "CHASI": "asa",
    "KIASI": "asa",
    "Kinyaturu": "rim",
    "BARIBA - TAMBERMA": "bba",
    "Ngam-Gam": "nmc",
    "Berber language": "ber",
    "LEBTHUR": "leb",
    "Ngumbo": "ngu",
    "NDOKWA": "ukw",
    "KWALE": "ukw",
    "KIKABWA": "cwa",
    "Sethepu": "ngh",
    "Phuthi": "ngh",
    "Zezuru": "sna",
    "Zarma/Songhay": "dje",
    "Wolof / Lebou": "wol",
    "VIGUE": "vig",
    "TSALA": "cll",
    "TILIBUNKA": "mnk",
    "TEHL": "mtl",
    "Silozi": "loz",
    "Setswana": "tsn",
    "SASARO": "sxs",
    "Rukwangali": "kwn",
    "Rugririku/Rumanyo": "diu",
    "RUBWISI": "tlj",
    "Otjiherero": "her",
    "Oshiwambo (Oshindonga/Oshikwanyama)": "kua",
    "Nama/Damara": "naq",
    "Masubia": "sub",
    "Malinké": "mlq",
    "Malagasy avec spécificité régionale": "mlg",
    "MENTIGNA": "tir",
    "Korekore": "sna",
    "KWAMASHI": "mho",
    "KINYIHA": "nih",
    "Ijaw": "ijo",
    "Haoussa": "hau",
    "Gedogna": "drs",
    "HALEBIGNA": "alw",
    "DIBBO": "dio",
    "Bété": "byf",
    "Béri béri (Kanouri)": "knc",
    "BETHE": "byf",
    "ANGAS": "anc",
    "AGNAGAN": "ayg",
    "Yacouba": "daf",
    "Luhya": "luh",
    "Nhaneca": "nyk",
    "Kikongo": "kon",
}

mapping_hf_iso3["iso_3_manual"] = mapping_hf_iso3["language_x"].map(manual_iso3_map)
mapping_hf_iso3["iso3"] = mapping_hf_iso3["iso3"].fillna(
    mapping_hf_iso3["iso_3_manual"]
)
mapping_hf_iso3 = mapping_hf_iso3[
    ["country", "language_x", "language_clean", "iso3"]
].rename(columns={"language_x": "language"})
mapping_hf_iso3

,country,language,language_clean,iso3
0,Angola,Chokwe,chokwe,cjk
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN
2,Angola,Kikongo,kikongo,kon
3,Angola,Kimbundu,kimbundu,kmb
4,Angola,LINGALA,lingala,lin
...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por
728,Zimbabwe,Shona,shona,sna
729,Zimbabwe,Tonga,tonga,NaN
730,Zimbabwe,Venda,venda,ven


> At this stage, we have 399 ISO-639-3 rows that are unmatched. We now use an LLM to attempt another round of matching language names. While this is not the best solution, it is the quickest way to get some proxy values assigned to facilitate search on Huggingface. We only use values that the LLM reports as confident. 

In [470]:
# extracting unmatched iso3 observations
unmatched_iso3_hf = mapping_hf_iso3[mapping_hf_iso3["iso3"].isna()]
unmatched_iso3_hf = unmatched_iso3_hf.sort_values(["country", "language"])
unmatched_iso3_hf

,country,language,language_clean,iso3
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN
6,Angola,Nganguela,nganguela,NaN
8,Angola,Oshiwambo/Oshikwanhama,oshiwambo/oshikwanhama,NaN
11,Benin,ALOGBA,alogba,NaN
12,Benin,Ajagbé,ajagbé,NaN
...,...,...,...,...
717,Zambia,Tokaleya,tokaleya,NaN
718,Zambia,Tonga,tonga,NaN
720,Zambia,Ushi,ushi,NaN
726,Zimbabwe,Ndebele,ndebele,NaN


> Note: convert following blocks back to Python code blocks to run them and query the LLM.

load_dotenv()
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("LITELLM_TOKEN"),
    base_url="https://litellm.oit.duke.edu/v1",
)

response = client.responses.create(model="gpt-5.2", input="ping")

print(response.output[0].content[0].text)



OUTFILE = IN_USE / "llm_iso_mapping.csv"

batch_size = 20
rows = list(unmatched_iso3_hf.itertuples(index=False))
results = []

hf_reference = "\n".join(
    hf_iso_3[["language"]]
    .dropna()
    .drop_duplicates()
    .sort_values("language")["language"]
    .tolist()
)

for i in range(0, len(rows), batch_size):
    batch = rows[i : i + batch_size]

    batch_text = "\n".join(
        [f"{r.country}|{r.language}|{r.language_clean}" for idx, r in enumerate(batch)]
    )


    prompt = f"""
You are matching Afrobarometer survey language labels to Hugging Face ISO-639-3 language names.

You may ONLY choose hf_language from this Hugging Face ISO-639-3 language list:
{hf_reference}

Input format:
country|afro_language|language_clean

Input rows:
{batch_text}

Rules:
- Return the same input_index.
- Copy country exactly from the input.
- Copy afro_language exactly from the input.
- Match afro_language/language_clean to ONE exact hf_language from the Hugging Face list.
- Do NOT return ISO codes.
- Do NOT invent language names.
- If uncertain, leave hf_language blank.
- Do NOT shift columns.
- Do NOT include a header.
- Do NOT use "|" inside any field.

Confidence:
HIGH = exact match or widely known synonym.
MEDIUM = plausible spelling/local variant with some ambiguity.
LOW = weak or uncertain.
BLANK = no appropriate match.

Return exactly {len(batch)} rows using this format:
country|afro_language|hf_language|confidence|notes
"""


    try:
        response = client.responses.create(
            model="gpt-5.2",
            input=prompt,
            temperature=0,
        )
        output = response.output_text.strip()

    except Exception as e:
        output = f"ERROR: {e}"

    results.append(
        {
            "batch_start": i,
            "batch_size": len(batch),
            "llm_output": output,
        }
    )

    pd.DataFrame(results).to_csv(OUTFILE, index=False)


> On my device, the above process took ~3 mins and cost around $0.4 worth of tokens.  

In [472]:
# converting LLM output into structured format
LLM_RESULT = IN_USE / "llm_iso_mapping.csv"
llm_results_raw = pd.read_csv(LLM_RESULT)

rows = []

for text in llm_results_raw["llm_output"]:
    for line in text.strip().split("\n"):
        if line.lower().startswith("country|"):
            continue

        parts = line.split("|", 5)

        if len(parts) < 6:
            parts += [""] * (6 - len(parts))

        rows.append(parts)

llm_results_parsed = pd.DataFrame(
    rows,
    columns=[
        "country",
        "afro_language",
        "hf_language",
        "iso3",
        "confidence",
        "notes",
    ],
)
llm_results_parsed

,country,afro_language,hf_language,iso3,confidence,notes
0,Angola,Fiote/Ibinda,Fiote,MEDIUM,Fiote is a Kikongo cluster variety; Ibinda not...,
1,Angola,Kikongo,Kongo,HIGH,Direct match to Kongo,
2,Angola,Nganguela,,BLANK,No clear exact match in provided HF list,
3,Angola,Nhaneca,,BLANK,No clear exact match in provided HF list,
4,Angola,Oshiwambo/Oshikwanhama,Kuanyama,HIGH,Oshikwanyama corresponds to Kuanyama,
...,...,...,...,...,...,...
394,Zambia,Tokaleya,,BLANK,Not found as exact name in provided HF list,
395,Zambia,Tonga,Tonga (Zambia),HIGH,Direct match to Tonga (Zambia),
396,Zambia,Ushi,,BLANK,Not found as exact name in provided HF list,
397,Zimbabwe,Ndebele,North Ndebele,HIGH,Zimbabwe Ndebele corresponds to North Ndebele,


In [473]:
# focusing only on confident responses
llm_results_high = llm_results_parsed.loc[
    llm_results_parsed["iso3"].str.upper().isin(["HIGH"]),
]

llm_results_high["lang_clean"] = (
    llm_results_high["afro_language"].str.strip().str.lower()
)
llm_results_high

,country,afro_language,hf_language,iso3,confidence,notes,lang_clean
1,Angola,Kikongo,Kongo,HIGH,Direct match to Kongo,,kikongo
4,Angola,Oshiwambo/Oshikwanhama,Kuanyama,HIGH,Oshikwanyama corresponds to Kuanyama,,oshiwambo/oshikwanhama
5,Benin,ALOGBA,Logba,HIGH,Direct match to Logba,,alogba
8,Benin,Ditamari,Ditammari,HIGH,Spelling variant of Ditammari,,ditamari
9,Benin,Fongbé,Fon,HIGH,Fongbe corresponds to Fon,,fongbé
...,...,...,...,...,...,...,...
386,Zambia,Bisa,Bisa,HIGH,Direct match in list,,bisa
388,Zambia,IIa,Ila,HIGH,Direct match to Ila,,iia
393,Zambia,Senga,Nsenga,HIGH,Direct match to Nsenga,,senga
395,Zambia,Tonga,Tonga (Zambia),HIGH,Direct match to Tonga (Zambia),,tonga


In [474]:
hf_name_to_iso3 = (
    hf_iso_3[["language", "iso3"]]
    .dropna()
    .drop_duplicates()
    .set_index("language")["iso3"]
    .to_dict()
)

llm_results_high["iso3_llm"] = llm_results_high["hf_language"].map(hf_name_to_iso3)
llm_results_high = llm_results_high[["lang_clean", "iso3_llm"]]
llm_results_high = llm_results_high.drop_duplicates(subset=["lang_clean"])

In [475]:
# updating country language list with LLM results
mapping_hf_iso3 = mapping_hf_iso3.merge(
    llm_results_high,
    left_on="language_clean",
    right_on="lang_clean",
    how="left",
    validate="many_to_one",
)

mapping_hf_iso3

,country,language,language_clean,iso3,lang_clean,iso3_llm
0,Angola,Chokwe,chokwe,cjk,NaN,NaN
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN,NaN
2,Angola,Kikongo,kikongo,kon,kikongo,kon
3,Angola,Kimbundu,kimbundu,kmb,NaN,NaN
4,Angola,LINGALA,lingala,lin,NaN,NaN
...,...,...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por,NaN,NaN
728,Zimbabwe,Shona,shona,sna,NaN,NaN
729,Zimbabwe,Tonga,tonga,NaN,tonga,toi
730,Zimbabwe,Venda,venda,ven,NaN,NaN


In [476]:
# fill missing iso3 with LLM-derived iso3
mapping_hf_iso3["iso3"] = mapping_hf_iso3["iso3"].fillna(mapping_hf_iso3["iso3_llm"])

# keep clean final columns
mapping_hf_iso3 = mapping_hf_iso3[["country", "language", "language_clean", "iso3"]]
mapping_hf_iso3

,country,language,language_clean,iso3
0,Angola,Chokwe,chokwe,cjk
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN
2,Angola,Kikongo,kikongo,kon
3,Angola,Kimbundu,kimbundu,kmb
4,Angola,LINGALA,lingala,lin
...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por
728,Zimbabwe,Shona,shona,sna
729,Zimbabwe,Tonga,tonga,toi
730,Zimbabwe,Venda,venda,ven


> At this stage, we have 200 ISO-639-3 rows that are unmatched. Further matching is unlikely at this stage.

In [ ]:
mapping_hf_iso1
mapping_hf_iso1_values = mapping_hf_iso1[mapping_hf_iso1["iso1"].notna()]
mapping_hf_iso1_values

In [477]:
mapping_combined = mapping_hf_iso3.merge(
    mapping_hf_iso1_values,
    on=["country", "language", "language_clean"],
    how="left",
    validate="many_to_one",
)
mapping_combined

,country,language,language_clean,iso3,iso1
0,Angola,Chokwe,chokwe,cjk,NaN
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN
2,Angola,Kikongo,kikongo,kon,NaN
3,Angola,Kimbundu,kimbundu,kmb,NaN
4,Angola,LINGALA,lingala,lin,ln
...,...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por,pt
728,Zimbabwe,Shona,shona,sna,sn
729,Zimbabwe,Tonga,tonga,toi,to
730,Zimbabwe,Venda,venda,ven,ve


In [478]:
# assiging back to the country languages list
country_languages = country_languages.merge(
    mapping_combined, on=["country", "language", "language_clean"], how="left"
)

country_languages

,country,language,language_clean,iso3,iso1
0,Angola,Chokwe,chokwe,cjk,NaN
1,Angola,Fiote/Ibinda,fiote/ibinda,NaN,NaN
2,Angola,Kikongo,kikongo,kon,NaN
3,Angola,Kimbundu,kimbundu,kmb,NaN
4,Angola,LINGALA,lingala,lin,ln
...,...,...,...,...,...
727,Zimbabwe,Portuguese,portuguese,por,pt
728,Zimbabwe,Shona,shona,sna,sn
729,Zimbabwe,Tonga,tonga,toi,to
730,Zimbabwe,Venda,venda,ven,ve


> At this stage, only 27 of 657 unique languages remain unmatched to ISO 639-3 codes. These account for just 0.06% of total responses when weighted using the survey sampling weights (combined). Given that this is (<1%), we proceed with the current mapping for the sake of continuing our analysis.

## 3. Merging Datasets

In [479]:
# creating country-language weight values
afro_lang_pop = afro.groupby(
    ["country", "language"], observed=True, as_index=False
).agg(language_weight=("weight", "sum"))
afro_lang_pop["country_language_share"] = (
    afro_lang_pop["language_weight"]
    / afro_lang_pop.groupby("country")["language_weight"].transform("sum")
) * 100
afro_lang_pop["language_clean"] = afro_lang_pop["language"].str.strip().str.lower()
afro_lang_pop

/var/folders/xl/fk37c10573g2b3yn7hpp8pv80000gn/T/ipykernel_8254/3418889650.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  / afro_lang_pop.groupby("country")["language_weight"].transform("sum")


,country,language,language_weight,country_language_share,language_clean
0,Angola,Chokwe,62.472624,5.526175,chokwe
1,Angola,Fiote/Ibinda,4.986178,0.441065,fiote/ibinda
2,Angola,Kikongo,24.477445,2.165215,kikongo
3,Angola,Kimbundu,4.686125,0.414523,kimbundu
4,Angola,LINGALA,0.746128,0.066001,lingala
...,...,...,...,...,...
727,Zimbabwe,Portuguese,0.358554,0.035498,portuguese
728,Zimbabwe,Shona,660.295048,65.370843,shona
729,Zimbabwe,Tonga,10.349992,1.024675,tonga
730,Zimbabwe,Venda,6.470331,0.640579,venda


In [480]:
# inserting ISO-639-3 and ISO-639-1 values back
afro_lang_pop = afro_lang_pop.merge(
    country_languages, on=["country", "language", "language_clean"], how="left"
)

afro_lang_pop

,country,language,language_weight,country_language_share,language_clean,iso3,iso1
0,Angola,Chokwe,62.472624,5.526175,chokwe,cjk,NaN
1,Angola,Fiote/Ibinda,4.986178,0.441065,fiote/ibinda,NaN,NaN
2,Angola,Kikongo,24.477445,2.165215,kikongo,kon,NaN
3,Angola,Kimbundu,4.686125,0.414523,kimbundu,kmb,NaN
4,Angola,LINGALA,0.746128,0.066001,lingala,lin,ln
...,...,...,...,...,...,...,...
727,Zimbabwe,Portuguese,0.358554,0.035498,portuguese,por,pt
728,Zimbabwe,Shona,660.295048,65.370843,shona,sna,sn
729,Zimbabwe,Tonga,10.349992,1.024675,tonga,toi,to
730,Zimbabwe,Venda,6.470331,0.640579,venda,ven,ve


In [482]:
# number of languages with no ISO 693-3 codes assigned at this stage
unmatched_iso3_afro_lang = afro_lang_pop[
    afro_lang_pop["iso3"].isna() & afro_lang_pop["iso1"].isna()
]

In [483]:
# Weighted share
no_iso_share = (
    unmatched_iso3_afro_lang["language_weight"].sum()
    / afro_lang_pop["language_weight"].sum()
) * 100
print(no_iso_share)

10.236409566063033


In [484]:
matched_afro_lang_pop = afro_lang_pop[
    afro_lang_pop["iso3"].notna() | afro_lang_pop["iso1"].notna()
]

In [485]:
# Weighted share
matched_iso_share = (
    matched_afro_lang_pop["language_weight"].sum()
    / afro_lang_pop["language_weight"].sum()
) * 100
print(matched_iso_share)

89.76359043393695


In [486]:
unmatched = unmatched_iso3_afro_lang[["country", "language", "country_language_share"]]
matched = matched_afro_lang_pop[
    ["country", "language", "country_language_share", "iso3", "iso1"]
]

> At this stage, we are left with 195 unmapped iso codes, representing 10% of respondents whose language could not be mapped to a ISO-639-3 code, after adjusting for survey weights. We assume that these languages will, consequently, be not found in the Huggingface database because a lookup using a consistent identifier will not be possible. 

In [488]:
# we now add population counts from the Joshua project dataset to our overall list
# key assumption made here that the "max" value from the Joshua Project (JP) is representative
# however, the max value is a value at the level of the globe as per the JP methodology
# so this is not a true picture of the number of speakers in Africa, esp for global languages like English, French, Arabic
# we assume that the values for Africa-only languages are representative enough as they are only spoken in Africa
# shrinking pop_count to afrobm_list

pop_counts = pop_counts[pop_counts["country"].isin(afrobm_countries)]
pop_iso = pop_counts.groupby("iso3", as_index=False)["speakers"].max()
afro_matched = matched.merge(pop_iso[["iso3", "speakers"]], on="iso3", how="left")

In [489]:
afro.to_csv(CLEAN / "afro_lang_pop.csv", index=False)
afro_matched.to_csv(CLEAN / "afro_matched.csv", index=False)
unmatched.to_csv(CLEAN / "afro_unmatched.csv", index=False)